# Thermoelectric data feasibility check (DFT + literature-mined experimental data)

This notebook fetches the two datasets I couldn't reach from my sandbox (figshare/Dryad are blocked
there, but should be reachable fine from Colab's network) and filters them for your chosen thermoelectric
family, so you get a full picture alongside the Starrydata2 numbers already pulled:

1. **Ricci et al. ab initio electronic transport database** — your DFT base-equation (`a`) source.
   Paper: https://www.nature.com/articles/sdata201785
2. **ChemDataExtractor-mined thermoelectric literature database** — an independent experimental source
   that already has a `Process` field extracted (sintering/annealing/milling/pressure), separate from
   Starrydata2. Paper: https://www.nature.com/articles/s41597-022-01752-1

Set `FAMILY` below to `"half-heusler"` or `"skutterudite"`, then **Runtime > Run all**.

At the end, download the exported CSVs (left sidebar > Files) and send them back in the chat with
Claude — it'll do the same cross-checking against the Starrydata2 numbers it already produced.


In [ ]:
FAMILY = "half-heusler"  # or "skutterudite" — change this and re-run all cells


## Setup

In [ ]:
import re
import zipfile
import io
import os
import requests
import pandas as pd

pd.set_option("display.max_columns", 50)
os.makedirs("data", exist_ok=True)


## Composition-matching heuristics

Rough regex filters for half-Heusler (ABX, 1:1:1, A = early transition/rare-earth metal,
B = late transition metal, X = Sn/Sb/Bi/As/Ge) and skutterudite (Co/Rh/Ir/Fe-Sb or -As, ~1:3 or
filled 1:4:12) compositions. These are heuristics, not authoritative structure classification —
sanity-check a sample of matches by eye, and prefer an explicit structure-type/space-group field
if either dataset provides one instead of relying on composition string matching alone.

In [ ]:
HH_A = r"(Ti|Zr|Hf|V|Nb|Ta|Sc|Y|La|Ce|Er|Lu|Dy|Ho|Tm|Yb|Gd|Tb)"
HH_B = r"(Ni|Co|Fe|Pd|Pt|Rh|Ru)"
HH_X = r"(Sn|Sb|Bi|As|Ge)"
HH_PATTERN = re.compile(HH_A + r".*" + HH_B + r".*" + HH_X)
SKUTT_PATTERN = re.compile(r"(Co|Rh|Ir|Fe|Ni)\d*\.?\d*(Sb|As|P)\d*\.?\d*")

def filter_family(df, composition_col, family):
    comp = df[composition_col].astype(str)
    if family == "half-heusler":
        mask = comp.apply(lambda s: bool(HH_PATTERN.search(s)))
    elif family == "skutterudite":
        mask = comp.apply(lambda s: bool(SKUTT_PATTERN.search(s)) and "Sb" in s)
    else:
        raise ValueError(family)
    return df[mask]


## Part 1 — Ricci et al. ab initio electronic transport database (DFT base-equation source)

Downloads the tabular BoltzTraP/Materials-Project export from figshare and unzips it. If the
ndownloader link below ever changes, grab the current one from the dataset page:
https://figshare.com/articles/dataset/ricci_boltztrap_mp_tabular/14701110


In [ ]:
url = "https://figshare.com/ndownloader/articles/14701110/versions/1"
r = requests.get(url, timeout=120)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    z.extractall("data/ricci_boltztrap")
    print(z.namelist())


In [ ]:
# Inspect what we actually got — figshare zips vary in structure/format.
for f in os.listdir("data/ricci_boltztrap"):
    print(f)


In [ ]:
# Once you see the actual filename above, load it and print columns so we know
# what the formula/composition column and the carrier-concentration/temperature
# columns are actually called.
# Example (EDIT the filename):
# ricci_df = pd.read_csv("data/ricci_boltztrap/<actual_filename>.csv")
# print(ricci_df.columns.tolist())
# print(ricci_df.head())


In [ ]:
# Once you know the composition/formula column name, filter for the chosen family
# and inspect the n (carrier concentration) / T grid it's tabulated on.
# Example (EDIT column names to match what you found above):
# ricci_family = filter_family(ricci_df, "formula", FAMILY)
# print(f"{FAMILY}: {len(ricci_family)} DFT entries")
# print("carrier concentration range:", ricci_family["N"].min(), "to", ricci_family["N"].max())
# print("temperature range:", ricci_family["T"].min(), "to", ricci_family["T"].max())
# ricci_family.to_csv(f"data/ricci_{FAMILY.replace(\'-\',\'\')}.csv", index=False)


## Part 2 — ChemDataExtractor-mined thermoelectric literature database (experimental, with a `Process` field)

Downloads the auto-extracted database (22,805 records from ~60,843 papers) from figshare. Current
link: https://doi.org/10.6084/m9.figshare.19658787


In [ ]:
url2 = "https://figshare.com/ndownloader/articles/19658787/versions/1"
r2 = requests.get(url2, timeout=180)
r2.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r2.content)) as z:
    z.extractall("data/thermoelectricsdb")
    print(z.namelist())


In [ ]:
for f in os.listdir("data/thermoelectricsdb"):
    print(f)


In [ ]:
# EDIT the filename once you see it above, then inspect columns —
# look specifically for a "Process" (or similarly named) field and a
# chemical-name/formula column.
# cde_df = pd.read_csv("data/thermoelectricsdb/<actual_filename>.csv")
# print(cde_df.columns.tolist())
# print(cde_df.head())


In [ ]:
# Filter for the chosen family and check how the Process field breaks down —
# this tells you directly whether your 2-3 candidate synthesis routes
# (e.g. SPS / hot pressing / melt spinning) show up as distinguishable labels.
# Example (EDIT column names):
# cde_family = filter_family(cde_df, "chemical_name_or_formula_col", FAMILY)
# print(f"{FAMILY}: {len(cde_family)} literature-mined records")
# print(cde_family["Process"].value_counts())
# cde_family.to_csv(f"data/cde_{FAMILY.replace(\'-\',\'\')}.csv", index=False)


## Part 3 — download the results

Run this to zip everything in `data/` so you can grab it from the Files sidebar (folder icon on the
left) and upload the CSVs back into the chat with Claude for cross-checking against the Starrydata2
numbers already pulled.


In [ ]:
import shutil
shutil.make_archive("thermoelectric_data_export", "zip", "data")
print("Wrote thermoelectric_data_export.zip — download it from the Files sidebar on the left.")
